In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v3").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


# Phase 2 · PubMed searches and abstracts

This notebook shows the selected organism groups, taxonomy inputs, cache/release locations, pending searches, and a lower-bound ETA before any NCBI request. The named `run_phase2` function searches one taxon at a time, fetches each PMID once, appends raw cache records, and checkpoints normalized Parquet releases. Reruns reuse cached queries and articles.

In [2]:
from pathlib import Path
import inspect
import pandas as pd
from IPython.display import display
from pathogen_atlas_v3.abstracts.pubmed import resolve_ncbi_email, _ncbi_key_from_keychain, acquire_pubmed
from pathogen_atlas_v3.common.config import selected_organism_groups
from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.pipelines.phase2_pubmed import phase2_preflight, run_phase2

CONFIG_PATH = "configs/default.yaml"
CONFIG = load_yaml(CONFIG_PATH)
GROUPS = selected_organism_groups(CONFIG)
TAXONOMY_ROOT = Path(CONFIG["paths"]["taxonomy_release"])
SPECIES = pd.read_parquet(TAXONOMY_ROOT / "species.parquet")
ALIASES = pd.read_parquet(TAXONOMY_ROOT / "aliases.parquet")
print("Selected groups:", GROUPS)
print("NCBI contact available:", bool(resolve_ncbi_email()))
print("NCBI API key available in environment or Keychain:", bool(_ncbi_key_from_keychain()))
print("Workers:", CONFIG["pubmed"]["max_workers"], "| request delay:", CONFIG["pubmed"]["request_delay_seconds"], "| retmax:", CONFIG["pubmed"]["retmax"])
display(SPECIES.loc[SPECIES["organism_group"].isin(GROUPS)].groupby("organism_group").size().rename("taxonomy species").to_frame())
print("\nImplementation called by run_phase2:\n")
print(inspect.getsource(run_phase2))

Selected groups: ['viruses']
NCBI contact available: True
NCBI API key available in environment or Keychain: True
Workers: 10 | request delay: None | retmax: 10000


,taxonomy species
organism_group,
viruses,17554



Implementation called by run_phase2:

def run_phase2(config_path: str = "configs/default.yaml", *, max_calls: int | None = None, max_fetch_batches: int | None = None, log: bool = True):
    config = load_yaml(config_path)
    species = pd.read_parquet(config["paths"]["taxonomy_release"] + "/species.parquet")
    aliases = pd.read_parquet(config["paths"]["taxonomy_release"] + "/aliases.parquet")
    settings, paths = config["pubmed"], config["paths"]
    groups = selected_organism_groups(config)
    root = Path(paths.get("pubmed_domain_root", Path(paths["pubmed_release"]).parent / "domains"))
    request_limit = max_calls if max_calls is not None else settings.get("max_calls")
    fetch_limit = max_fetch_batches if max_fetch_batches is not None else settings.get("max_fetch_batches")
    progress_every = int(config.get("logging", {}).get("progress_every", 100))
    results = []
    remaining_calls = None if request_limit is None else int(request_limit)
    for group in groups:
        g

## Preflight · no network calls

Check the pending query count and ETA before enabling the run. The ETA estimates only ESearch pacing; retries, PubMed result volumes, and abstract retrieval add time. Query and fetch progress logs print current completion and ETA at the interval configured by `logging.progress_every`.

In [3]:
PREFLIGHT = phase2_preflight(CONFIG_PATH)
display(PREFLIGHT)
CACHE_AND_RELEASE = pd.DataFrame([{"group": row["organism_group"], "species": row["species"], "cached queries": row["cached_queries"], "pending queries": row["pending_queries"], "cache": row["cache_path"], "release": str(Path(CONFIG["paths"]["pubmed_domain_root"]) / row["organism_group"] / "release")} for row in PREFLIGHT["per_group"]])
display(CACHE_AND_RELEASE)

{'status': 'ready',
 'selected_groups': ['viruses'],
 'per_group': [{'organism_group': 'viruses',
   'species': 17554,
   'cached_queries': 16053,
   'pending_queries': 1504,
   'cache_path': 'data/02_pubmed/domains/viruses/cache/pubmed_search_results.jsonl'}],
 'pending_queries': 1504,
 'request_delay_seconds': 0.2,
 'estimated_search_floor_minutes': 5.0,
 'retmax_per_query': 10000,
 'estimated_search_note': 'Lower-bound ESearch time only; excludes retries, throttling, and abstract retrieval.'}

,group,species,cached queries,pending queries,cache,release
0,viruses,17554,16053,1504,data/02_pubmed/domains/viruses/cache/pubmed_se...,data/02_pubmed/domains/viruses/release


## Run Phase 2

Set `MAX_QUERY_CALLS` and `MAX_FETCH_BATCHES` to small values for a bounded run. Set them to `None` for the full selected-domain acquisition. Interrupting the notebook is safe: completed work is cached and the same call resumes.

In [ ]:
RUN_PHASE = True
MAX_QUERY_CALLS = None  # full selected-domain search; cache makes reruns resumable
MAX_FETCH_BATCHES = None  # fetch every missing abstract batch
LOG_PROGRESS = True

if RUN_PHASE:
    if not resolve_ncbi_email():
        raise RuntimeError("Set NCBI_EMAIL or configure the local Git author email before NCBI requests.")
    MATCHES, ABSTRACTS = run_phase2(CONFIG_PATH, max_calls=MAX_QUERY_CALLS, max_fetch_batches=MAX_FETCH_BATCHES, log=LOG_PROGRESS)
    print(f"Phase 2 returned {len(MATCHES):,} selected species–PMID rows and {len(ABSTRACTS):,} unique abstract records.")
else:
    print("Preflight only. Set RUN_PHASE=True to run the selected organism groups.")

Phase 2 · viruses: 17,554 taxa; aliases 54,297; cache data/02_pubmed/domains/viruses/cache; release data/02_pubmed/domains/viruses/release.
PubMed search: 17,554 species rows; 16,053 cached queries; 1,504 pending.
PubMed search: 100/1,504; ETA 8.0 min.
PubMed search: 200/1,504; ETA 5.8 min.
PubMed search: 300/1,504; ETA 5.6 min.
PubMed search: 400/1,504; ETA 5.2 min.
